In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-5-mini")

#### Problem 

In [ ]:
text = "My name is Manya Bhutada./" \
" my age is 21 and i live in Bangalore"

res = llm.invoke(f"Please extract the name, age, and location from the following text: {text}")
print(res.content)
print(type(res.content))     #its type is string, what if we want to get in any other format


#### Solution: Structured output

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="Only first name of the person")
    age: int = Field(ge = 0, le = 100)
    city: str = Field(description="City where the person lives")

structured_llm = llm.with_structured_output(Person)
res = structured_llm.invoke(f"Please extract the name, age, and location from the following text: {text}")
print(res)
print(type(res))  
print(res.model_dump())
print(res.model_dump_json())
print(type(res.model_dump_json()))  
print(type(res.model_dump()))


#### typing library

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Movie(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(ge=2000, le=2026)
    genre: str = Field(description="Genre of the movie")

class MovieList(BaseModel):
    movies: List[Movie] = Field(description="List of movies")

movie_llm = llm.with_structured_output(MovieList)
res = movie_llm.invoke("Please provide a list of 3 bollywood movies released after 2000 with their title, year, and genre.")
print(res.model_dump())

#### how structured output help with communication between two LLMs

In [17]:
#llm1: extract information in structured format

class trip_details(BaseModel):
    destination: str = Field(description="Destination of the trip")
    duration: int = Field(ge=1, le=30, description="Duration of the trip in days")
    budget: float = Field(ge=1000, le=100000, description="Budget for the trip in indian rupees")

extractor = llm.with_structured_output(trip_details)
res = extractor.invoke("I want to go on a trip to Manali for 5 days with a budget of 50000 rupees. Please provide the details in a structured format.")
trip_data = res.model_dump()
print(trip_data)

prompt = f"""
I want to go on a trip for {trip_data['duration']} to {trip_data['destination']} with a budget of {trip_data['budget']} rupees.
give me a proper iterary for the trip
"""
result = llm.invoke(prompt)
print(result.content)

{'destination': 'Manali', 'duration': 5, 'budget': 50000.0}
Quick question before I make a detailed plan: do you mean a trip for 5 days or a trip for 5 people? And is the 50,000 INR budget total or per person? Also, what city will you start from (Delhi, Chandigarh, etc.) and roughly when do you want to travel?

While you reply, here’s a ready-to-use sample itinerary I’ll prepare now assuming:
- Trip length = 5 days / 4 nights
- Starting city = Delhi (common route). Tell me if yours is different and I’ll adapt.
- Budget = ₹50,000 total for one person (you can tell me if it’s for a group).

Sample 5-day Manali itinerary (Delhi → Manali → Delhi) with approximate costs and notes

Summary budget estimate (per person, approximate)
- Delhi → Manali roundtrip (Volvo AC sleeper): ₹2,000–3,000
  OR flight to Bhuntar roundtrip: ₹6,000–12,000 (if faster)
- Accommodation (4 nights, mid-range hotel/private room): ₹6,000–10,000
- Food (5 days): ₹1,500–3,000
- Local taxis / day trips (shared/private):